# Function Calling

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
OPENWEATHER_API_KEY=os.getenv('OPENWEATHER_API_KEY')

## Function (tool) 준비

In [3]:
import requests
city_name = "Busan"
url = f'https://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={OPENWEATHER_API_KEY}&units=metric'
response = requests.get(url)
data = response.json()

In [4]:
data

{'coord': {'lon': 129.0403, 'lat': 35.1028},
 'weather': [{'id': 803,
   'main': 'Clouds',
   'description': 'broken clouds',
   'icon': '04d'}],
 'base': 'stations',
 'main': {'temp': 12.99,
  'feels_like': 11.65,
  'temp_min': 12.99,
  'temp_max': 12.99,
  'pressure': 1022,
  'humidity': 50,
  'sea_level': 1022,
  'grnd_level': 1017},
 'visibility': 10000,
 'wind': {'speed': 4.12, 'deg': 140},
 'clouds': {'all': 75},
 'dt': 1773645223,
 'sys': {'type': 1,
  'id': 8086,
  'country': 'KR',
  'sunrise': 1773610440,
  'sunset': 1773653482},
 'timezone': 32400,
 'id': 1838524,
 'name': 'Busan',
 'cod': 200}

In [5]:
response

<Response [200]>

In [6]:
weather_info={}

if response.status_code==200:   #status_code==200인지 확인하는 과정
    # 정상 응답
    description=data['weather'][0]['description']
    temperature=data['main']['temp']
    temp_fells_like=data['main']['feels_like']
    humidity=data['main']['humidity']
    weather_info = {
        'city': city_name,
        'description':description,
        'temperature':temperature,
        'temperature_fells_like':temp_fells_like,
        'humidity': humidity
    }

else: 
    # 오류 (실패)
    weather_info = {
        'city': city_name,
        'description':'Not Found',
        'temperature':'Not Found',
        'temperature_fells_like':'Not Found',
        'humidity': 'Not Found'
    }
weather_info

{'city': 'Busan',
 'description': 'broken clouds',
 'temperature': 12.99,
 'temperature_fells_like': 11.65,
 'humidity': 50}

In [7]:
import json

def get_current_weather(city_name='Seoul',units='metric'):
    '''
    OpenWeater API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수
    '''

    url = f'https://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={OPENWEATHER_API_KEY}&units={units}'
    response = requests.get(url)
    data = response.json()

    weather_info={}

    if response.status_code==200:   #status_code==200인지 확인하는 과정
        # 정상 응답
        description=data['weather'][0]['description']
        temperature=data['main']['temp']
        temp_fells_like=data['main']['feels_like']
        humidity=data['main']['humidity']
        weather_info = {
            'city': city_name,
            'description':description,
            'temperature':temperature,
            'temperature_fells_like':temp_fells_like,
            'humidity': humidity
        }
    
    else: 
        # 오류 (실패)
        weather_info = {
            'city': city_name,
            'description':'Not Found',
            'temperature':'Not Found',
            'temperature_fells_like':'Not Found',
            'humidity': 'Not Found'
        }
    return json.dumps(weather_info)

In [8]:
get_current_weather.__doc__

'\n    OpenWeater API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수\n    '

In [9]:
get_current_weather()

'{"city": "Seoul", "description": "overcast clouds", "temperature": 11.76, "temperature_fells_like": 10.04, "humidity": 40}'

In [10]:
tools_to_execute={
    'get_current_weather': get_current_weather
}

## LLM 준비

In [ ]:
import openai

client = openai.OpenAI()

def run_conversation(user_prompt, model='gpt-4o-mini'): # 모델명 오타 수정
    messages = [ # 변수명 복수형 권장
        {'role': 'system', 'content': '당신은 친절한 챗봇입니다.'},
        {'role': 'user', 'content': user_prompt},
    ]
    
    tools = [
        {
            'type': 'function',
            'function': {
                'name': 'get_current_weather',
                'description': "현재 날씨 정보를 가져옵니다.", # 예시용
                'parameters': {
                    'type': 'object',
                    'properties': {
                        'city_name': {
                            'type': 'string',
                            'description': '도시이름(필수값). 영어로 작성 (예: Seoul)'
                        }, 
                        'units': {
                            'type': 'string',
                            'description': 'metric(섭씨) 또는 imperial(화씨)',
                            'enum': ['metric', 'imperial']
                        }
                    },
                    'required': ['city_name']
                }
            }
        }
    ]

    # response 변수에 결과 할당
    response = client.chat.completions.create(
        model=model,
        messages=messages, # 매개변수명 확인
        tools=tools
    )
    
    response_message = response.choices[0].message
    print(response_message)
    response_tool_calls=response_message.tool_calls

    if response_tool_calls:
        # 함수 호출(NOn이 아닌 경우)
        messages.append(response_message)

        for tool_call in response_tool_calls:
            function_name = tool_call.function.name
            print(f'[tool] {function_name} 호출...')
            func_to_exe=tools_to_execute[function_name]
            func_args=json.loads(tool_call.function.arguments)
            func_responose=func_to_exe(**func_args)

            messages.append({
                'role': 'tool',
                'tool_call_id':tool_call.id,
                'name': function_name,
                'content': func_responose
            })

            client.chat.completions.create(
                model=model,
                messages=messages
            )
            return response.choices[0].message.content
        
    else:
        # Non인 경우
        return response_message.content
    
    #아직 에이전트 구조를 세팅한 것이 아니라 function만 넣은 상태

In [18]:
run_conversation('서울 날씨 어때?')

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_OKyIpRdPKVLXycj3jiBOHVdN', function=Function(arguments='{"city_name":"Seoul","units":"metric"}', name='get_current_weather'), type='function')])
[tool] get_current_weather 호출...


In [19]:
run_conversation('점심 메뉴는 어때?')

ChatCompletionMessage(content='점심 메뉴로는 여러 가지가 좋습니다! 여기 몇 가지 아이디어를 소개할게요:\n\n1. **김치볶음밥** - 간단하고 맛있어요!\n2. **비빔국수** - 시원하고 상큼한 맛이 좋죠.\n3. **불고기 덮밥** - 달콤한 소스와 고기가 어우러져 맛있습니다.\n4. **샐러드 볼** - 건강한 선택으로 다양한 채소와 단백질을 가득 담아보세요.\n5. **치킨 샌드위치** - 간편하고 기름지지 않아서 인기 있어요.\n\n어떤 메뉴가 마음에 드시나요?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


'점심 메뉴로는 여러 가지가 좋습니다! 여기 몇 가지 아이디어를 소개할게요:\n\n1. **김치볶음밥** - 간단하고 맛있어요!\n2. **비빔국수** - 시원하고 상큼한 맛이 좋죠.\n3. **불고기 덮밥** - 달콤한 소스와 고기가 어우러져 맛있습니다.\n4. **샐러드 볼** - 건강한 선택으로 다양한 채소와 단백질을 가득 담아보세요.\n5. **치킨 샌드위치** - 간편하고 기름지지 않아서 인기 있어요.\n\n어떤 메뉴가 마음에 드시나요?'

---

In [22]:
import openai

client = openai.OpenAI()

def run_conversation(user_prompt, model='gpt-4o-mini'): 
    message = [
        {'role': 'system', 'content': '당신은 친절한 챗봇입니다. 사용자의 요구를 분석해 직접 대답하거나, 주어진 함수를 이용해 필요한 정보를 제공합니다.'},
        {'role': 'user', 'content': user_prompt},
    ]
    tools = [
        {
            'type': 'function',
            'function': {
                'name': 'get_current_weather',
                'description': "현재 날씨 정보를 가져옵니다.", 
                'parameters': {
                    'type': 'object',
                    'properties': {
                        'city_name': { 
                            'type': 'string',
                            'description': '''도시이름(필수값). 반드시 영어로 작성하세요.
                                            - 변환예시:
                                            - 서울 -> Seoul
                                            - 충남, 충청남도 -> Chungcheongnam-do
                                            - 경남, 경상남도 -> Gyeongsangnam-do
                                            - 전남, 전라남도 -> Jeollanam-do
                                            - 부산 -> Busan
                                            '''
                        },
                        'units': {
                            'type': 'string',
                            'description': '''온도단위를 설정하는 문자열
                                            - metric(기본값: 섭씨, 미터)
                                            - imperial(화씨, 야드)''',
                            'enum': ['metric', 'imperial']
                        }
                    },
                    'required': ['city_name']
                }
            }
        }
    ]
    # response 변수에 결과 할당
    response = client.chat.completions.create(
        model=model,
        messages=messages, # 매개변수명 확인
        tools=tools
    )
    
    response_message = response.choices[0].message
    print(response_message)
    response_tool_calls=response_message.tool_calls

    if response_tool_calls:
        # 함수 호출(NOn이 아닌 경우)
        messages.append(response_message)

        for tool_call in response_tool_calls:
            function_name = tool_call.function.name
            print(f'[tool] {function_name} 호출...')
            func_to_exe=tools_to_execute[function_name]
            func_args=json.loads(tool_call.function.arguments)
            func_responose=func_to_exe(**func_args)

            messages.append({
                'role': 'tool',
                'tool_call_id':tool_call.id,
                'name': function_name,
                'content': func_responose
            })

            client.chat.completions.create(
                model=model,
                messages=messages
            )
            return response.choices[0].message.content
        
    else:
        # Non인 경우
        return response_message.content
    
    #아직 에이전트 구조를 세팅한 것이 아니라 function만 넣은 상태

In [23]:
run_conversation('서울 날씨 어때?')

NameError: name 'messages' is not defined